# Causilo on Amazon SageMaker

Causilo is a tabular foundation model. You send one table: the rows whose answer you already know, followed by the rows you want predicted with the answer left blank. Causilo fills in the blanks in a single inference call, with no training run.

This notebook deploys the Causilo model package from AWS Marketplace in your own AWS account, predicts a held-out part of two public datasets on a real-time endpoint, runs the classification table through a batch transform job, and deletes what it created. The container runs with network isolation, so your data stays in your account.

Use through AWS Marketplace is governed by the listing's end user license agreement, the [Causilo License v1.0](https://huggingface.co/nums-ai/causilo/blob/2edc3d8e84149271ab9d87c61be2b4860dc3a938/LICENSE): non-commercial research, testing and evaluation. Commercial or production use of Causilo or its outputs, or offering it as a hosted or API service, paid or free, needs a separate license from Nums AI Inc. (api@nums.world).

## Before you start

1. Subscribe to Causilo on AWS Marketplace.
2. Run this notebook in SageMaker Studio or a SageMaker notebook instance, in a Region where Causilo is offered. The model package, the execution role and the S3 bucket are found (the bucket is created if it does not exist) from the Region and the notebook's own role.
3. Your account needs quota for one `ml.g6e.xlarge` for endpoint usage and one `ml.g5.2xlarge` for transform job usage. To use other instance types, change them in the next cell.
4. The endpoint and the transform job are billed while they run. If any cell from step 2 on fails, run step 5 before you leave. The model, the endpoint and the transform job are tagged `causilo-demo` with the run's name.

In [ ]:
%pip install -q pyarrow pandas scikit-learn

In [ ]:
import io, json, time
import boto3
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError, WaiterError

REGION = boto3.session.Session().region_name
if not REGION:
    raise SystemExit("No AWS Region configured. Set AWS_DEFAULT_REGION.")

# Causilo model package ARN per Region, as published on AWS Marketplace.
MODEL_PACKAGES = {
}
MODEL_PACKAGE_ARN = MODEL_PACKAGES.get(REGION)
if not MODEL_PACKAGE_ARN:
    raise SystemExit(
        f"No Causilo model package listed here for {REGION}. "
        f"Offered in: {', '.join(sorted(MODEL_PACKAGES)) or '(none yet)'}. "
        "Copy the ARN for this Region from the product's configuration page on AWS Marketplace into MODEL_PACKAGES."
    )

INSTANCE_TYPE = "ml.g6e.xlarge"         # real-time endpoint, NVIDIA L40S
BATCH_INSTANCE_TYPE = "ml.g5.2xlarge"   # batch transform, NVIDIA A10G; ml.m5.2xlarge also works (CPU, slower)
ROLE_ARN = None                         # set this if the role cannot be found automatically

sts = boto3.client("sts", region_name=REGION)
ACCOUNT = sts.get_caller_identity()["Account"]

def execution_role():
    """The IAM role this notebook runs as, which SageMaker also uses to run the model."""
    arn = sts.get_caller_identity()["Arn"]
    if ":assumed-role/" not in arn:
        raise SystemExit("Run this in SageMaker Studio or a SageMaker notebook instance, or set ROLE_ARN to a SageMaker execution role.")
    role = arn.split(":assumed-role/")[1].split("/")[0]
    try:
        iam = boto3.client("iam", config=Config(connect_timeout=5, retries={"total_max_attempts": 1}))
        return iam.get_role(RoleName=role)["Role"]["Arn"]
    except (ClientError, BotoCoreError):
        # Without iam:GetRole (or without network access to IAM), rebuild the ARN.
        # SageMaker's default roles live under /service-role/; for a role under another path, set ROLE_ARN.
        path = "service-role/" if role.startswith("AmazonSageMaker-ExecutionRole") else ""
        return f"arn:aws:iam::{ACCOUNT}:role/{path}{role}"

ROLE_ARN = ROLE_ARN or execution_role()
S3_BUCKET = f"sagemaker-{REGION}-{ACCOUNT}"   # SageMaker's default bucket

sm = boto3.client("sagemaker", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION, config=Config(read_timeout=70, retries={"total_max_attempts": 1}))
s3 = boto3.client("s3", region_name=REGION)
try:
    s3.head_bucket(Bucket=S3_BUCKET, ExpectedBucketOwner=ACCOUNT)
except ClientError as e:
    code = e.response["Error"]["Code"]
    if code not in ("404", "NoSuchBucket"):
        raise SystemExit(f"S3 bucket {S3_BUCKET} is not usable ({code}): it belongs to another account, or this role lacks s3:ListBucket.")
    kw = {} if REGION == "us-east-1" else {"CreateBucketConfiguration": {"LocationConstraint": REGION}}
    s3.create_bucket(Bucket=S3_BUCKET, **kw)

def gone(e):
    """True if a SageMaker call failed because the resource no longer exists."""
    msg = e.response["Error"]["Message"].lower()
    return e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound") and ("could not find" in msg or "not found" in msg)

RUN_FILE = ".causilo-demo-run"   # remembers the run's name across kernel restarts, for step 5
print(REGION, MODEL_PACKAGE_ARN, ROLE_ARN, S3_BUCKET, sep="\n")

## 1. Build the input tables

Two public datasets that ship with scikit-learn, so nothing is downloaded:

- [Breast Cancer Wisconsin (Diagnostic)](https://doi.org/10.24432/C5DW2B): 569 breast mass samples, 30 measurements of cell nuclei, benign or malignant. W. Wolberg, O. Mangasarian, N. Street and W. Street, UCI Machine Learning Repository, [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- Diabetes: 442 patients, 10 baseline variables, disease progression one year later. Efron, Hastie, Johnstone and Tibshirani (2004), "Least Angle Regression", *Annals of Statistics* 32(2), 407–499.

These datasets only demonstrate the request format. Causilo is not a medical device and is not intended for diagnostic use.

For each, a random 100 rows become the query rows with the answer removed, and the rest are the context rows with the answer kept. Causilo sees the context rows and fills in the query rows; the true answers stay in this notebook for scoring.

The request is one Parquet file. Rows with a blank target are the query rows; this notebook puts them after the context rows. Parameters go in the file's key-value metadata under `causilo-params`:

- `model`: `causilo-clf` for classification or `causilo-reg` for regression
- `target`: the name of the target column
- `output_type`: for classification `preds` (labels) or `probas` (class probabilities, the default); for regression `mean` (the default), `median` or `quantiles` (with a `quantiles` list such as `[0.1, 0.5, 0.9]`)

The classification request is also in the cookbook repository as `aws-sagemaker/sample-input.parquet`.

In [ ]:
from sklearn.datasets import load_breast_cancer, load_diabetes

def split(frame, n_query=100, seed=0):
    idx = np.random.default_rng(seed).permutation(len(frame))
    return frame.iloc[idx[n_query:]].reset_index(drop=True), frame.iloc[idx[:n_query]].reset_index(drop=True)

def to_parquet(context, query, params):
    frame = pd.concat([context, query], ignore_index=True)
    frame[params["target"]] = frame[params["target"]].astype(object)
    frame.loc[len(context):, params["target"]] = None   # the answers Causilo fills in
    table = pa.Table.from_pandas(frame, preserve_index=False)
    table = table.replace_schema_metadata({"causilo-params": json.dumps(params)})
    buf = io.BytesIO()
    pq.write_table(table, buf)
    return buf.getvalue()

cancer = load_breast_cancer(as_frame=True)
clf_frame = cancer.frame.assign(target=cancer.target_names[cancer.target])   # "malignant" / "benign"
clf_context, clf_query = split(clf_frame)
clf_payload = to_parquet(clf_context, clf_query, {"model": "causilo-clf", "target": "target", "output_type": "preds"})

diabetes = load_diabetes(as_frame=True)
reg_context, reg_query = split(diabetes.frame)
reg_payload = to_parquet(reg_context, reg_query, {"model": "causilo-reg", "target": "target", "output_type": "mean"})

print(f"classification: {len(clf_context)} context + {len(clf_query)} query rows, {clf_frame.shape[1] - 1} features, {len(clf_payload):,} bytes")
print(f"regression:     {len(reg_context)} context + {len(reg_query)} query rows, {diabetes.frame.shape[1] - 1} features, {len(reg_payload):,} bytes")

## 2. Deploy a real-time endpoint

This takes three to four minutes.

In [ ]:
import os
if "name" in globals() or os.path.exists(RUN_FILE):
    raise SystemExit("An earlier run has not been cleaned up. Run step 5 first, then restart from here.")
name = f"causilo-demo-{int(time.time())}"
open(RUN_FILE, "w").write(name)
prefix = f"causilo-demo/{name}/"
tags = [{"Key": "causilo-demo", "Value": name}]
print("Resources for this run are named", name)

sm.create_model(
    ModelName=name,
    ExecutionRoleArn=ROLE_ARN,
    Containers=[{"ModelPackageName": MODEL_PACKAGE_ARN}],
    EnableNetworkIsolation=True,
    Tags=tags,
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": name,
        "InstanceType": INSTANCE_TYPE,
        "InitialInstanceCount": 1,
    }],
    Tags=tags,
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name, Tags=tags)
sm.get_waiter("endpoint_in_service").wait(EndpointName=name)
print("InService")

## 3. Predict

The endpoint is deleted at the end of this step, since the batch transform in step 4 does not need it.

In [ ]:
def predict(payload, n_query):
    response = runtime.invoke_endpoint(
        EndpointName=name,
        ContentType="application/vnd.apache.parquet",
        Accept="application/json",
        Body=payload,
    )
    result = json.loads(response["Body"].read())   # {"prediction": [...], "metadata": {...}}
    assert len(result["prediction"]) == n_query, result["metadata"]
    return result

try:
    clf = predict(clf_payload, len(clf_query))
    accuracy = np.mean(np.array(clf["prediction"]) == clf_query["target"].to_numpy())
    print(f"classification accuracy on {len(clf_query)} held-out rows: {accuracy:.3f}")
    print("first five:", clf["prediction"][:5], "true:", clf_query["target"].head().tolist())

    reg = predict(reg_payload, len(reg_query))
    pred, true = np.array(reg["prediction"], dtype=float), reg_query["target"].to_numpy()
    r2 = 1 - ((true - pred) ** 2).sum() / ((true - true.mean()) ** 2).sum()
    print(f"regression R² on {len(reg_query)} held-out rows: {r2:.3f}")
finally:
    try:
        sm.delete_endpoint(EndpointName=name)
        print("endpoint deleted")
    except ClientError as e:
        if not gone(e):
            raise

## 4. Batch transform

A real-time request is limited to 6 MB and 60 seconds. For larger tables, put one Parquet file per S3 object, up to 100 MB each, and run a batch transform job. Each output object holds the same JSON as the real-time response.

In [ ]:
s3.put_object(Bucket=S3_BUCKET, Key=f"{prefix}input/table.parquet", Body=clf_payload, ExpectedBucketOwner=ACCOUNT)

sm.create_transform_job(
    TransformJobName=name,
    ModelName=name,
    MaxConcurrentTransforms=1,
    MaxPayloadInMB=100,   # MaxConcurrentTransforms x MaxPayloadInMB must stay at or below 100
    ModelClientConfig={"InvocationsTimeoutInSeconds": 3600, "InvocationsMaxRetries": 0},
    TransformInput={
        "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{S3_BUCKET}/{prefix}input/"}},
        "ContentType": "application/vnd.apache.parquet",
        "SplitType": "None",
    },
    TransformOutput={"S3OutputPath": f"s3://{S3_BUCKET}/{prefix}output/", "Accept": "application/json"},
    TransformResources={"InstanceType": BATCH_INSTANCE_TYPE, "InstanceCount": 1},
    Tags=tags,
)
try:
    sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=name, WaiterConfig={"Delay": 60, "MaxAttempts": 120})
except WaiterError:
    pass
job = sm.describe_transform_job(TransformJobName=name)
if job["TransformJobStatus"] != "Completed":
    raise SystemExit(f"Transform job {job['TransformJobStatus']}: {job.get('FailureReason', '')} Run step 5 to stop it and clean up.")
out = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=f"{prefix}output/table.parquet.out", ExpectedBucketOwner=ACCOUNT)["Body"].read())
batch_accuracy = np.mean(np.array(out["prediction"]) == clf_query["target"].to_numpy())
print(f"batch classification accuracy: {batch_accuracy:.3f}")

## 5. Clean up

Stops the transform job if it is still running, and deletes the endpoint, the endpoint configuration, the model and this run's files in S3. Transform jobs cannot be deleted; the job record, the CloudWatch logs of the endpoint and the job, and the S3 bucket stay, at little or no cost.

In [ ]:
import os
if "name" not in globals():
    if not os.path.exists(RUN_FILE):
        raise SystemExit("Nothing to clean up.")
    name = open(RUN_FILE).read().strip()
    prefix = f"causilo-demo/{name}/"

errors = []
def attempt(call, **kw):
    try:
        call(**kw)
    except ClientError as e:
        if not gone(e):
            errors.append(f"{call.__name__}: {e}")

# The endpoint first: it is the part that is billed by the hour.
attempt(sm.delete_endpoint, EndpointName=name)
attempt(sm.delete_endpoint_config, EndpointConfigName=name)

try:
    status = sm.describe_transform_job(TransformJobName=name)["TransformJobStatus"]
    if status == "InProgress":
        sm.stop_transform_job(TransformJobName=name)
    if status in ("InProgress", "Stopping"):
        try:
            sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=name)
        except WaiterError:
            pass
except ClientError as e:
    if not gone(e):
        errors.append(f"transform job: {e}")

attempt(sm.delete_model, ModelName=name)

for page in s3.get_paginator("list_objects_v2").paginate(Bucket=S3_BUCKET, Prefix=prefix, ExpectedBucketOwner=ACCOUNT):
    keys = [{"Key": o["Key"]} for o in page.get("Contents", [])]
    if keys:
        resp = s3.delete_objects(Bucket=S3_BUCKET, Delete={"Objects": keys}, ExpectedBucketOwner=ACCOUNT)
        errors += [f"s3 {x['Key']}: {x['Code']}" for x in resp.get("Errors", [])]

if errors:
    raise SystemExit("Cleanup incomplete; fix and run this cell again:\n" + "\n".join(errors))
if os.path.exists(RUN_FILE):
    os.remove(RUN_FILE)
print("deleted", name)
del name

## More

- Data contract, parameters and error codes: https://docs.nums.world/reference (this documents the hosted API; on SageMaker, the limits in step 4 apply)
- Technical report: https://arxiv.org/abs/2609.22866
- Support: api@nums.world